In [ ]:
setwd("splicing")

library(plyr)
library(dplyr)
library(qvalue)
library(msigdbr)
library(data.table)

source("code/enrichment_analysis.R")

In [3]:
msigdb <- msigdbr(species="Mus musculus", category="C5")
sets <- tapply(msigdb$gene_symbol, INDEX=msigdb$gs_name, FUN=list)
sets <- sets[grep("^GO", names(sets))]

# Ctrl vs. CalTRAP SR

In [ ]:
experiment_path <- "data/rmats/ctrl_vs_caltrap_SR"

In [ ]:
# combine all detected splicing events to get background

all_files <- list.files(path=experiment_path, pattern="fromGTF\\.[^.]+\\.txt")
all_events_list <- lapply(all_files, function(x) {
    fread(file.path(experiment_path, x), data.table=FALSE)
})
all_events_df <- do.call(rbind.fill, all_events_list)
all_genes <- unique(all_events_df$geneSymbol)

In [ ]:
# combine singificant splicing events

signif_files <- list.files(path=experiment_path, pattern="filtered")

signif_events_list <- lapply(signif_files, function(x) {
    df <- fread(file.path(experiment_path, x), data.table=FALSE)
    df$Event_type <- unlist(strsplit(x, split=".", fixed=TRUE))[1]
    df
})
signif_events_df <- do.call(rbind.fill, signif_events_list)
signif_genes <- unique(signif_events_df$geneSymbol)

## enrichments for all significant events

In [9]:
enrich_df <- get_enrich(signif_genes, all_genes, sets)
head(enrich_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOCC_SYNAPSE,3.543316e-11,3.684695e-07
GOCC_NEURON_PROJECTION,3.922051e-09,2.039270e-05
GOBP_CELL_JUNCTION_ORGANIZATION,1.630757e-08,5.652747e-05
GOBP_NEURON_DIFFERENTIATION,2.491422e-08,6.477075e-05
GOMF_SEQUENCE_SPECIFIC_DNA_BINDING,3.633167e-08,7.556261e-05
GOCC_AXON,1.882288e-07,3.262320e-04
GOBP_NEUROGENESIS,3.140280e-07,4.506446e-04
GOBP_CELL_PART_MORPHOGENESIS,3.466830e-07,4.506446e-04
GOBP_POSITIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS,4.810798e-07,5.188860e-04


## enrichments for retained intron events

### note: negative values corresponds to more retained introns in caltrap samples

In [ ]:
RI_cal_df <- signif_events_df[(signif_events_df$Event_type == "RI") & (signif_events_df$IncLevelDifference < 0),]
RI_cal_genes <- unique(RI_cal_df$geneSymbol)
enrich_RI_cal_df <- get_enrich(RI_cal_genes, all_genes, sets)

[1] 50


### Positive values corresponds to more retained introns in control samples

In [12]:
RI_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "RI") & (signif_events_df$IncLevelDifference > 0),]
RI_ctrl_genes <- unique(RI_ctrl_df$geneSymbol)
print(length(RI_ctrl_genes))

[1] 29


In [13]:
enrich_RI_ctrl_df <- get_enrich(RI_ctrl_genes, all_genes, sets)
head(enrich_RI_ctrl_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_NEGATIVE_REGULATION_OF_OXIDATIVE_STRESS_INDUCED_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY,0.0004315405,1
GOBP_POSITIVE_REGULATION_OF_GLUCONEOGENESIS,0.0004315405,1
GOBP_RESPONSE_TO_OXYGEN_CONTAINING_COMPOUND,0.0006271147,1
GOBP_REGULATION_OF_OXIDATIVE_STRESS_INDUCED_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY,0.0009408539,1
GOBP_POSITIVE_REGULATION_OF_GLUCOSE_METABOLIC_PROCESS,0.0015277161,1
GOBP_REGULATION_OF_EXTRINSIC_APOPTOTIC_SIGNALING_PATHWAY_IN_ABSENCE_OF_LIGAND,0.0015277161,1


## Skipped exons

### Negative values corresponds to more skipped exons in CalTRAP samples

In [14]:
SE_cal_df <- signif_events_df[(signif_events_df$Event_type == "SE") & (signif_events_df$IncLevelDifference < 0),]
SE_cal_genes <- unique(SE_cal_df$geneSymbol)
print(length(SE_cal_genes))

[1] 365


In [15]:
enrich_SE_cal_df <- get_enrich(SE_cal_genes, all_genes, sets)
head(enrich_SE_cal_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOCC_NEURON_PROJECTION,1.078855e-11,1.121901e-07
GOCC_SYNAPSE,1.114080e-09,5.792658e-06
GOBP_CELL_JUNCTION_ORGANIZATION,6.863519e-08,2.379125e-04
GOBP_CELL_PART_MORPHOGENESIS,5.108207e-07,1.328006e-03
GOBP_NEURON_DIFFERENTIATION,9.695998e-07,1.738348e-03
GOBP_CELLULAR_COMPONENT_MORPHOGENESIS,1.002990e-06,1.738348e-03
GOMF_PROTEIN_DOMAIN_SPECIFIC_BINDING,1.292100e-06,1.919507e-03
GOBP_MICROTUBULE_ANCHORING_AT_CENTROSOME,2.709334e-06,3.521796e-03
GOCC_AXON,3.079858e-06,3.558605e-03


### Positive values corresponds to more skipped exons in control samples

In [16]:
SE_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "SE") & (signif_events_df$IncLevelDifference > 0),]
SE_ctrl_genes <- unique(SE_ctrl_df$geneSymbol)
print(length(SE_ctrl_genes))

[1] 309


In [17]:
enrich_SE_ctrl_df <- get_enrich(SE_ctrl_genes, all_genes, sets)
head(enrich_SE_ctrl_df, 10)

,Pval,P.adj
,<dbl>,<dbl>
GOMF_NUCLEOSIDE_TRIPHOSPHATASE_REGULATOR_ACTIVITY,4.207631e-05,0.3084672
GOCC_PLASMA_MEMBRANE_REGION,8.278914e-05,0.3084672
GOBP_CARDIAC_MUSCLE_CELL_ACTION_POTENTIAL,1.397475e-04,0.3084672
GOBP_CELL_JUNCTION_ORGANIZATION,1.464107e-04,0.3084672
GOBP_HISTONE_PHOSPHORYLATION,1.990857e-04,0.3084672
GOCC_BASAL_PART_OF_CELL,2.484260e-04,0.3084672
GOCC_I_BAND,2.752983e-04,0.3084672
GOBP_CYTOSKELETON_ORGANIZATION,3.019894e-04,0.3084672
GOBP_MEMBRANE_DEPOLARIZATION_DURING_CARDIAC_MUSCLE_CELL_ACTION_POTENTIAL,3.491350e-04,0.3084672


## Mutally exclusive exons

In [18]:
MXE_df <- signif_events_df[signif_events_df$Event_type == "MXE",]
MXE_genes <- unique(MXE_df$geneSymbol)
print(length(MXE_genes))

[1] 96


In [19]:
enrich_MXE_df <- get_enrich(MXE_genes, all_genes, sets)
head(enrich_MXE_df)

,Pval,P.adj
,<dbl>,<dbl>
GOMF_HIGH_VOLTAGE_GATED_CALCIUM_CHANNEL_ACTIVITY,3.668272e-05,0.2455474
GOBP_NEGATIVE_REGULATION_OF_TRANSCRIPTION_BY_RNA_POLYMERASE_II,4.722520e-05,0.2455474
GOBP_NEGATIVE_REGULATION_OF_BIOSYNTHETIC_PROCESS,2.872027e-04,0.7939329
GOBP_NEGATIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS,3.285218e-04,0.7939329
GOBP_POSITIVE_REGULATION_OF_CARBOHYDRATE_METABOLIC_PROCESS,3.817352e-04,0.7939329
GOMF_NEUROPEPTIDE_BINDING,4.631208e-04,0.8026654


## Alternative 3' splice site

### Negative values corresponds to longer 3' end in CalTRAP samples

In [20]:
A3SS_cal_df <- signif_events_df[(signif_events_df$Event_type == "A3SS") & (signif_events_df$IncLevelDifference < 0),]
A3SS_cal_genes <- unique(A3SS_cal_df$geneSymbol)
print(length(A3SS_cal_genes))

[1] 62


In [21]:
enrich_A3SS_cal_df <- get_enrich(A3SS_cal_genes, all_genes, sets)
head(enrich_A3SS_cal_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_POSITIVE_REGULATION_OF_CELLULAR_BIOSYNTHETIC_PROCESS,6.560843e-05,0.6596189
GOBP_MRNA_SPLICE_SITE_SELECTION,2.271479e-04,0.6596189
GOBP_REGULATION_OF_HEART_RATE_BY_CHEMICAL_SIGNAL,2.886623e-04,0.6596189
GOBP_POSITIVE_REGULATION_OF_RNA_METABOLIC_PROCESS,2.942542e-04,0.6596189
GOBP_REGULATION_OF_CELLULAR_MACROMOLECULE_BIOSYNTHETIC_PROCESS,3.171550e-04,0.6596189
GOBP_CENTRAL_NERVOUS_SYSTEM_NEURON_DIFFERENTIATION,8.347929e-04,1.0000000


### Positive values corresponds to longer 3' end in control samples


In [22]:
A3SS_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "A3SS") & (signif_events_df$IncLevelDifference > 0),]
A3SS_ctrl_genes <- unique(A3SS_ctrl_df$geneSymbol)
print(length(A3SS_ctrl_genes))

[1] 52


In [23]:
enrich_A3SS_ctrl_df <- get_enrich(A3SS_ctrl_genes, all_genes, sets)
head(enrich_A3SS_ctrl_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_HISTONE_METHYLATION,7.678522e-06,0.06690732
GOCC_NUCLEAR_BODY,1.286803e-05,0.06690732
GOBP_PROTEIN_METHYLATION,2.552446e-05,0.08847628
GOBP_METHYLATION,7.225478e-05,0.18784436
GOMF_TRANSITION_METAL_ION_TRANSMEMBRANE_TRANSPORTER_ACTIVITY,1.670858e-04,0.34750498
GOCC_SPECTRIN_ASSOCIATED_CYTOSKELETON,2.028030e-04,0.35149144


## Alternative 5' splice site

### Negative values corresponds to longer 5' end in CalTRAP samples

In [24]:
A5SS_cal_df <- signif_events_df[(signif_events_df$Event_type == "A5SS") & (signif_events_df$IncLevelDifference < 0),]
A5SS_cal_genes <- unique(A5SS_cal_df$geneSymbol)
print(length(A5SS_cal_genes))

[1] 51


In [25]:
enrich_A5SS_cal_df <- get_enrich(A5SS_cal_genes, all_genes, sets)
head(enrich_A5SS_cal_df)

,Pval,P.adj
,<dbl>,<dbl>
GOCC_CENTRIOLAR_SUBDISTAL_APPENDAGE,3.816595e-06,0.03968877
GOMF_ATP_DEPENDENT_CHROMATIN_REMODELER_ACTIVITY,2.724170e-04,0.90383415
GOBP_POSITIVE_REGULATION_OF_INTRINSIC_APOPTOTIC_SIGNALING_PATHWAY,3.284250e-04,0.90383415
GOCC_CILIARY_TRANSITION_FIBER,4.648168e-04,0.90383415
GOBP_TRANSLATIONAL_ELONGATION,5.498007e-04,0.90383415
GOBP_MICROTUBULE_ANCHORING_AT_CENTROSOME,5.796615e-04,0.90383415


### Positive values corresponds to longer 5' end in control samples


In [26]:
A5SS_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "A5SS") & (signif_events_df$IncLevelDifference > 0),]
A5SS_ctrl_genes <- unique(A5SS_ctrl_df$geneSymbol)
print(length(A5SS_ctrl_genes))

[1] 41


In [27]:
enrich_A5SS_ctrl_df <- get_enrich(A5SS_ctrl_genes, all_genes, sets)
head(enrich_A5SS_ctrl_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_POSITIVE_REGULATION_OF_TRANSCRIPTION_BY_RNA_POLYMERASE_II,3.428216e-05,0.290971
GOMF_SEQUENCE_SPECIFIC_DNA_BINDING,6.669336e-05,0.290971
GOCC_IKAPPAB_KINASE_COMPLEX,8.394201e-05,0.290971
GOBP_POSITIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS,1.732643e-04,0.370030
GOMF_NUCLEAR_RECEPTOR_ACTIVITY,1.855106e-04,0.370030
GOCC_TRANSCRIPTION_REGULATOR_COMPLEX,2.134994e-04,0.370030
